# Data Analysis 

This notebook loads and analyses results.

In [1]:
# importing necessary packages

using DelimitedFiles
using JLD2
using DataFrames
using Statistics
using Images   
using Plots
using FileIO
using ImageIO
using Colors
using LinearAlgebra
using TiffImages
using CSV
using Contour

## Functions

Four functions were used in this model: "meshgrid", "findnearest". "ModelFromImage", and "diffusion". The purposes of thee functions are listed below:

meshgrid: function to find the index of the nearest value in a vector to assign a units value \
findnearest: function to take difference to find which pixels correspond to which grid cell \
ModelFromImage: function to create a model from the image imported \
diffusion: function to calculate contrubution of diffusion to heat throughout model \
find_isotherm_line \
find_isotherm_distances 

In [ ]:
# function to find the index of the nearest value in a vector to assign a units value************************

function meshgrid(x, y) 
    X = repeat(x', length(y), 1)          # transpose x to make it a row vector
    Y = repeat(y, 1, length(x))           # repeat y to make it a column vector
    return X, Y
end

# x and y: 1D vectors of x and y coords resp
# X and Y: 2D matrices of x and y coords resp

In [ ]:
# findnearest: function to take difference to find which pixels correspond to which grid cell*****************

function findnearest(vec,val) 
    _, idx = findmin(abs.(vec.-val))      # take difference to find closest index
    return idx
end

# vec: 1D vector of values
# val: value to find the nearest index for
# idx: index of nearest value in vec

In [ ]:
# function to create a model from the image imported**********************************************************

function ModelFromImage(file, W, Nx)

    img = load(file)                                                # load the image file
    p,q = size(img)                                                 # get image dimensions

    # setting colours to identify rock types and assign units
    yellow_rgb = [1, 230/255, 128/255]                              # yellow pixel rgb values 
    orange_rgb = [1, 127/255, 39/255]                               # orange pixel rgb values
    red_rgb    = [237/255, 28/255, 36/255]                          # red pixel rgb values
    white_rgb  = [1, 1, 1]                                          # white pixel rgb values

    img_units  = zeros(Int,p,q)                                     # 2D matrix made to be same size as image
    

    # for loop going through every pixel in image
    for i in 1:p, j in 1:q                                         

        pixel = img[i, j]                                           # get pixel color at (i,j)
        rgb = pixel.color                                           # extract RGB values from pixel color

        # extract red, green, and blue components from pixel colour
        r = red(rgb)                                                # red function
        g = green(rgb)                                              # green function
        b = blue(rgb)                                               # blue function
        
        pixel_vec = [r,g,b]                                         # create vector of RGB values

        # check if pixel color is close to one of the defined colors
        if norm(pixel_vec .- yellow_rgb) < 0.01                     # assess each colour one by one
            img_units[i, j] = 1                                     # yellow unit
        elseif norm(pixel_vec .- orange_rgb) < 0.01
            img_units[i, j] = 2                                     # orange unit
        elseif norm(pixel_vec .- red_rgb) < 0.01
            img_units[i, j] = 3                                     # red unit
        else
            img_units[i, j] = 4                                     # white/ outliers unit
        end 
    end

    D   = W*p/q                                                     # calculate depth based on image height and width proportion
    Nz  = floor(Int,Nx*p/q)                                         # calculate number of grid points in z based on image, and take floor to ensure integer value

    # ensure all variables are integers
    q   = Int(q)
    p   = Int(p)
    Nx  = Int(Nx)
    Nz  = Int(Nz)

    # image coordinates for creating grid of image
    hi       = W / q                                               # image grid spacing
    xci      = range(hi/2, stop = W - hi/2, length = q)            # x coordinates for image grid
    zci      = range(hi/2, stop = D - hi/2, length = p)            # same for z
    Xci, Zci = meshgrid(xci, zci)                                  # create meshgrid for image coordinates

    # model or "target" coordinates for creating grid of model
    h     = W/Nx                                                   # model grid spacing
    xc    = range(h/2, stop = W - h/2; length = Nx)                # x coordinates for model grid
    zc    = range(h/2, stop = D - h/2; length = Nz)                # same for z
    Xc,Zc = meshgrid(xc,zc)                                        # create meshgrid for model coordinates

    
    # create empty array for interpolated image units
    img_inter = Array{Int}(undef, Nz, Nx)                          
    

    # for loop going through each grid cell
    for i in 1:Nz
        for j in 1:Nx                                              
            
            x = xc[j]                                              # get x coordinate of pixel
            z = zc[i]                                              # same for z coords
            p_j = findnearest(xci, x)                              # find nearest x coordinate in image grid
            p_i = findnearest(zci, z)                              # same for z coords

            img_inter[i, j] = img_units[p_i, p_j]                  # assign interpolated image unit to model grid
        end
    end


    units = Int.(img_inter)                                        # convert interpolated image units to integers
    return units, Nz, D
end

# file:   path to image file
# W:      width of model in metres
# Nx:     number of grid points in x direction
# units:  matrix of rock unit numbers
# D:      depth of model i metres
# Nz:     number of grid poits in z direction

In [ ]:
function find_isotherm_distances(T, z_cc, T_high, T_low)
    
    Nz, Nx = size(T)
    
    depth_isotherm = fill(NaN, Nx)                        # initialise vectors to store depths for isotherms
    depth_air = fill(NaN, Nx)
    
    # loop through temperature matrix
    for j in 1:Nx

        idx_isotherm = findall(T[:, j] .>= T_high)        # find row where temp is at least T_high
        idx_air = findall(T[:, j] .<= T_low)              # find row where temp is at most T_low (air temp)
        
        
        if !isempty(idx_isotherm)

            depth_isotherm[j] = z_cc[first(idx_isotherm)] # find the minimum depth to T_high

        end

        if !isempty(idx_air)

            depth_air[j] = z_cc[last(idx_air)]           # find depth to air contour
        
        end
    end

    
    distance_vector = depth_isotherm .- depth_air         # find distance between isotherm and air contour
    
   
    if isempty(distance_vector)
        println("Warning: cannot calculate distance.")    # check the distance has values and return warning message if there are NaN vals
        return NaN
    else
        return minimum(distance_vector)                   # find the minimum distance from the valid distances.
    end

end

In [ ]:
function find_isotherm_line2(T, z_cc, T_high, T_low)

    Nz, Nx = size(T)
    

    depth_25C = fill(NaN, Nx)
    depth_8C = fill(NaN, Nx)
    

    for j in 1:Nx

        idx_25C = findall(T[:, j] .>= T_high)
        idx_8C = findall(T[:, j] .<= T_low)
        
        if !isempty(idx_25C)
            depth_25C[j] = z_cc[first(idx_25C)]
        end
        
        if !isempty(idx_8C)
            depth_8C[j] = z_cc[last(idx_8C)]
        end
    end
    
    return minimum(depth_25C .- depth_8C )
end

## Load Data 

In [ ]:
W    = 4e3; # width of the domain, should match image  [m]
dx   = 20; # spacing between each grid point - resolution  [m]
Nx   = Int64(W/dx); # number of column in x direction ie 100  
file = "//campus.gla.ac.uk/isi/stud-file/Documents/Julia/image.tiff"
units, Nz, D = ModelFromImage(file, W, Nx) # set outputs of function as variables


x_cc = range(dx/2, stop = W - dx/2, length = Nx)  # Horizontal
z_cc = range(dx/2, stop = D - dx/2, length = Nz)  # Vertical

In [ ]:
# load data

#base
data_T_sim1 = load("sim_results_Diff_Base.jld2")
T_sim1 = data_T_sim1["T_final"]


#Hr
data_T_sim2 = load("sim_results_Diff_incr_Hr-2.jld2")
T_sim2 = data_T_sim2["T_final"]

data_T_sim3 = load("sim_results_Diff_decr_Hr-3.jld2")
T_sim3 = data_T_sim3["T_final"]


#Cp
data_T_sim4 = load("sim_results_Diff_incr_Cp-4-new.jld2")
T_sim4 = data_T_sim4["T_final"]

data_T_sim5 = load("sim_results_Diff_decr_Cp-5-new.jld2")
T_sim5 = data_T_sim5["T_final"]


#sigma
data_T_sim6 = load("sim_results_Diff_incr_sigma-6-new.jld2")
T_sim6 = data_T_sim6["T_final"]

data_T_sim7 = load("sim_results_Diff_decr_sigma-7.jld2")
T_sim7 = data_T_sim7["T_final"]


#rho
data_T_sim8 = load("sim_results_Diff_incr_rho-8.jld2")
T_sim8 = data_T_sim8["T_final"]

data_T_sim9 = load("sim_results_Diff_decr_rho-9.jld2")
T_sim9 = data_T_sim9["T_final"]


#dTdz
data_T_sim10 = load("sim_results_Diff_incr_dTdz-10.jld2")
T_sim10 = data_T_sim10["T_final"]

data_T_sim11 = load("sim_results_Diff_decr_dTdz-11.jld2")
T_sim11 = data_T_sim11["T_final"]

## Results: Temperature Stats

In [10]:
# create a dataframe to add results to

results = DataFrame(
    simulation = String[],
    max_T = Float64[],
    min_T = Float64[],
    avg_T = Float64[],
    max_diff_vs_base = Float64[],
    min_dist_25C = Float64[],
    min_dist_40C = Float64[]
)

In [ ]:
# set up arrays to be used in loops

Temps = [T_sim1, T_sim2, T_sim3, T_sim4, T_sim5, T_sim6, T_sim7, T_sim8, T_sim9, T_sim10, T_sim11]
Tempsno1 = [T_sim2, T_sim3, T_sim4, T_sim5, T_sim6, T_sim7, T_sim8, T_sim9, T_sim10, T_sim11]
sims = ["sim1", "sim2", "sim3", "sim4", "sim5","sim6", "sim7", "sim8", "sim9", "sim10", "sim11"]
numbers = ["2", "3", "4", "5", "6", "7", "8", "9", "10", "11"] 


# ensure T matrix is correct dimensions 
T_base = T_sim1
if size(T_base) != (length(z_cc), length(x_cc))
    T_base = T_base'
end


In [ ]:
# load in difference results: temperature difference from base across domain

T_differences = Dict{String, Matrix{Float64}}()
total_differences = Dict{String, Float64}()

# take away base temperature matrix and put into dictionary
for (T, sim) in zip(Tempsno1, numbers)
    
    T_difference_from_base = T - T_sim1
    T_differences[sim] = T_difference_from_base

end

#find the total temperature difference
for (sim_num, diff_matrix) in T_differences
   
    total_sum = sum(diff_matrix)
    total_differences[sim_num] = total_sum

end

In [ ]:
# set those differences as variables from dictionaries

T_diff_sim2 = T_differences["2"]
T_diff_sim3 = T_differences["3"]
T_diff_sim4 = T_differences["4"]
T_diff_sim5 = T_differences["5"]
T_diff_sim6 = T_differences["6"]
T_diff_sim7 = T_differences["7"]
T_diff_sim8 = T_differences["8"]
T_diff_sim9 = T_differences["9"]
T_diff_sim10 = T_differences["10"]
T_diff_sim11 = T_differences["11"]

In [ ]:
# change file and input names depending on which difference wanting to plot

gr()

# set limits to be used for colourbar in plot 
if maximum(T_diff_sim2)>0
    limits = (-maximum(T_diff_sim2), maximum(T_diff_sim2)) # find maximum if increases in temperature
else 
    limits = (minimum(T_diff_sim2), -minimum(T_diff_sim2)) # find minimum if decreases in temperature
end

p = plot(
    x_cc, 
    z_cc,
    T_diff_sim2; 
    st = :heatmap,
    aspect_ratio = 1,
    xlabel = "x [m]",
    ylabel = "z [m]",
    title = "Effect of Increasing Radiogenic Heating",
    colorbar_title = "Temperature [°C]",
    c = :balance,
    yflip = true,
    xlims = (0, 4000),
    fontfamily = "Times New Roman",
    size = (800, 600),
    clims = limits
)

contour!(p, x_cc, z_cc, T_diff_sim2, 
        levels=[0.001], 
        linecolor=[:black], 
        linewidth=[1], 
        label = false, colorbar=true)

savefig(p, "T_differences_incr_Hr-2.pdf")
p2

In [51]:
# creat loop to find maximum, minimum, mean temperatures and maximum difference for 25 and 40 degrees

for (T, sim) in zip(Temps, sims)

    max_T = maximum(T)
    min_T = minimum(T)
    avg_T = mean(T)
    diff = maximum(abs.(T .- T_base))
    
    min_dist_25 = find_isotherm_distances(T, collect(z_cc),25, 8.2) # 
    min_dist_40 = find_isotherm_distances(T, collect(z_cc),40, 8.2) 

    push!(results, (
        simulation = sim,
        max_T = max_T,
        min_T = min_T,
        avg_T = avg_T,
        max_diff_vs_base = diff,
        min_dist_25C = min_dist_25,
        min_dist_40C = min_dist_40 # Store the result in the new column
    ))

end

CSV.write("sensitivity_results.csv", results)

## Results: Isotherms

Making plots of isotherm positions for base model and geothermal gradient models.

In [ ]:
# make plots of isotherm lines 

gr()

depth_8C_results = Dict{String, Vector{Float64}}()
depth_25C_results = Dict{String, Vector{Float64}}()
depth_50C_results = Dict{String, Vector{Float64}}()
depth_100C_results = Dict{String, Vector{Float64}}()

# looping through to use isotherms as a contour for plots
for (T, sim) in zip(Temps, sims)

    T_r = round.(T)
    Nz, Nx = size(T_r)

    depth_8C = fill(NaN, Nx)
    depth_25C = fill(NaN, Nx)
    depth_50C = fill(NaN, Nx)
    depth_100C = fill(NaN, Nx)

    for j in 1:Nx

        idx_8C = findall(abs.(T_r[:, j] .- 8) .< 0.5)
        idx_25C = findall(abs.(T_r[:, j] .- 25) .< 0.5)
        idx_50C = findall(abs.(T_r[:, j] .- 50) .< 0.5)
        idx_100C = findall(abs.(T_r[:, j] .- 100) .< 0.5)
        
        if !isempty(idx_8C)
            depth_8C[j] = z_cc[last(idx_8C)]
        end
        
        if !isempty(idx_25C)
            depth_25C[j] = z_cc[first(idx_25C)]
        end

        if !isempty(idx_50C)
            depth_50C[j] = z_cc[first(idx_50C)]
        end

        if !isempty(idx_100C)
            depth_100C[j] = z_cc[first(idx_100C)]
        end
    end

    depth_8C_results[sim] = depth_8C
    depth_25C_results[sim] = depth_25C
    depth_50C_results[sim] = depth_50C
    depth_100C_results[sim] = depth_100C

end

sim1_depth_25C = depth_25C_results["sim1"]
sim2_depth_25C = depth_25C_results["sim2"]
sim3_depth_25C = depth_25C_results["sim3"]
sim10_depth_25C = depth_25C_results["sim10"]
sim11_depth_25C = depth_25C_results["sim11"]

sim1_depth_8C = depth_8C_results["sim1"]

plot(
    x_cc, 
    sim1_depth_25C, 
    label = "Base",
    lw = 2,
    color = :slateblue3,
    title = "25°C Isotherm Depths",
    xlabel = "x [m]",
    ylabel = "z [m]",
    size = (800, 410),
    legend = :outertopright,
    yflip = true, 
    ylim = (0, 1000),
    fontfamily = "Times New Roman"
)

# second line (sim2) 
plot!(
    x_cc, 
    sim10_depth_25C, 
    label = "Incr dT/dz",
    lw = 2,
    color = :turquoise3,
)

# third line (sim3) 
plot!(
    x_cc, 
    sim11_depth_25C, 
    label = "Decr dT/dz",
    lw = 2,
    color = :deeppink,
)

# surface contour
plot!(
    x_cc, 
    sim1_depth_8C, 
    label = " Surface ",
    lw = 2,
    color = :black
)

annotate!([
    (360, 220, text("A", 6, :red)),
    (1050, 220, text("B", 6, :red)),
    (1650, 220, text("C", 6, :red)),
    (2300, 220, text("D", 6, :red)),
    (2800, 220, text("E", 6, :red)),
    (3200, 220, text("F", 6, :red))])

savefig("25C_isotherm_depths_dTdz.pdf")

In [ ]:
sim1_depth_50C = depth_50C_results["sim1"]
sim10_depth_50C = depth_50C_results["sim10"]
sim11_depth_50C = depth_50C_results["sim11"]

plot(
    x_cc, 
    sim1_depth_50C, 
    label = "Base",
    lw = 2,
    color = :slateblue3,
    title = "50°C Isotherm Depths",
    xlabel = "x [m]",
    ylabel = "z [m]",
    size = (800, 550),
    legend = :outertopright,
    yflip = true, 
    ylim = (0, 1800),
    fontfamily = "Times New Roman"
)

# second line (sim2) 
plot!(
    x_cc, 
    sim10_depth_50C, 
    label = "Incr dT/dz",
    lw = 2,
    color = :turquoise,
)

# third line (sim3) 
plot!(
    x_cc, 
    sim11_depth_50C, 
    label = "Decr dT/dz",
    lw = 2,
    color = :deeppink,
)

# surface contour
plot!(
    x_cc, 
    sim1_depth_8C, 
    label = " Surface ",
    lw = 2,
    color = :black
)
annotate!([
    (360, 230, text("A", 6, :red)),
    (1050, 220, text("B", 6, :red)),
    (1650, 220, text("C", 6, :red)),
    (2300, 220, text("D", 6, :red)),
    (2800, 220, text("E", 6, :red)),
    (3200, 220, text("F", 6, :red))])

savefig("50C_isotherm_depths_dTdz.pdf")

In [ ]:
sim1_depth_100C = depth_100C_results["sim1"]
sim10_depth_100C = depth_100C_results["sim10"]
sim11_depth_100C = depth_100C_results["sim11"]

plot(
    x_cc, 
    sim1_depth_100C, 
    label = "Base",
    lw = 2,
    color = :slateblue3,
    title = "100°C Isotherm Depths",
    xlabel = "x [m]",
    ylabel = "z [m]",
    size = (800, 650),
    legend = :outertopright,
    yflip = true, 
    ylim = (0, 3500),
    fontfamily = "Times New Roman"
)

# second line (sim2) 
plot!(
    x_cc, 
    sim10_depth_100C, 
    label = "Incr dT/dz",
    lw = 2,
    color = :turquoise,
)

# third line (sim3) 
plot!(
    x_cc, 
    sim11_depth_100C, 
    label = "Decr dT/dz",
    lw = 2,
    color = :deeppink,
)

# surface contour
plot!(
    x_cc, 
    sim1_depth_8C, 
    label = " Surface ",
    lw = 2,
    color = :black
)

annotate!([
    (360, 230, text("A", 6, :red)),
    (1050, 220, text("B", 6, :red)),
    (1650, 220, text("C", 6, :red)),
    (2300, 220, text("D", 6, :red)),
    (2800, 220, text("E", 6, :red)),
    (3200, 220, text("F", 6, :red))])
savefig("100C_isotherm_depths_dTdz.pdf")

In [ ]:
# make isotherm results dataframe

# your isotherm values
iso_values = [25, 50, 100]

# pre-allocate a DataFrame to store results
results = DataFrame(
    Simulation = String[],
    Isotherm   = Float64[],
    Avg        = Float64[],
    Min        = Float64[],
    Max        = Float64[],
)

# loop over each simulation and isotherm
for (i, T_sim) in enumerate(Temps)
    for iso in iso_values
        vals = find_isotherm_line2(T_sim, z_cc, iso, 8.2)
        push!(results, (
            "T_sim$(i)",  # Simulation name
            iso,
            mean(vals),
            minimum(vals),
            maximum(vals),
        ))
    end
end

# write results to CSV
CSV.write("isotherm_results.csv", results)

# also print to screen if you like
println(results)